CHURN PREDICTION

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('data/Telco_Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.head().T #to see all columns

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = df.select_dtypes(include=['object','string','str']).columns

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [9]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce') #ignore errors and convert to NaN
df.totalcharges = df.totalcharges.fillna(0) #replace NaN with 0 but better to do dropna

#df = df.dropna(subset=['totalcharges'])

In [11]:
#check if churn variable is useful
#here we have yes or no but we need numbers
#convert yes to 1 and no to 0
df.churn = (df.churn == 'yes').astype(int)


In [14]:
from sklearn.model_selection import train_test_split

In [18]:
df_full_train,df_test = train_test_split(df,test_size=0.2,random_state=1)

df_train,df_val = train_test_split(df_full_train,test_size=0.25,random_state=1)
# 20% of 80%
len(df_train),len(df_val),len(df_test)

(4225, 1409, 1409)

In [19]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [ ]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

#to avoid using the variables when building the model

"del df_train['churn']\ndel df_val['churn']\ndel df_test['churn']"

In [23]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
df_full_train.isnull().sum() # no additional data preparation required

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [ ]:
df_full_train.churn.value_counts(normalize=True)#percentage of churn and not churn
#27% churn rate
#can also do mean because it is binary dataset

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [26]:
global_churn_rate = df_full_train.churn.mean()

In [28]:
numerical = ['tenure','monthlycharges','totalcharges'] #list of numerical attributes
categorical = ['gender','seniorcitizen','partner','dependents','phoneservice','multiplelines','internetservice','onlinesecurity','onlinebackup','deviceprotection','techsupport','streamingtv','streamingmovies','contract','paperlessbilling','paymentmethod'] #list of categorical attributes

In [ ]:
df_full_train[categorical].nunique

check the churn rates among different categories(male,female) etc
this gives us a better idea of the distribution

eg: churn_partner = df_full_train[df_full_train.partner=='yes'].churn().mean()

difference between male,female not prominent
difference between partner ,no partner is higher -> means it is more important for prediction

FEATURE IMPORTANCE

DIFFERENCE:
global-group => >0 more likely to churn 

RISK RATIO:
GROUP/GLOBAL => >1 more likely to churn

SELECT 
    GENDER,
    AVG(CHURN),
    AVG(CHURN)-GLOBAL_CHURN AS DIFF,
    AVG(CHURN)/GLOBAL_CHURN AS RISK

    FROM DATA 
    GROUP BY GENDER

In [33]:
from IPython.display import display

In [36]:
for c in categorical:
    df_group = df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk'] = df_group['mean'] / global_churn_rate
    display(df_group)

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208


,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


,mean,count,diff,risk
dependents,,,,
no,0.313760,3968,0.043792,1.162212
yes,0.165666,1666,-0.104302,0.613651


,mean,count,diff,risk
phoneservice,,,,
no,0.241316,547,-0.028652,0.893870
yes,0.273049,5087,0.003081,1.011412


...

,mean,count,diff,risk
streamingtv,,,,
no,0.342832,2246,0.072864,1.269897
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.302723,2167,0.032755,1.121328


,mean,count,diff,risk
streamingmovies,,,,
no,0.338906,2213,0.068938,1.255358
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.307273,2200,0.037305,1.138182


,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


it is difficult to look at all individually and we only get some intuition

we calculate mutual information

In [37]:
from sklearn.metrics import mutual_info_score

In [39]:
mutual_info_score(df_full_train['contract'],df_full_train['churn'])

0.0983203874041556

In [40]:
mutual_info_score(df_full_train['gender'],df_full_train['churn'])

0.0001174846211139946

this shows that contracct matters much more than gender

we can apply this to all variables

In [41]:
def mutual_info_churn_score(series):
    return mutual_info_score(series,df_full_train['churn'])

In [43]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

Correlation

wrt tenure-> the longer a customer stays with us the lesser the likelihood of leaving

In [44]:
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

ONE HOT ENCODING

Imagine

gender , contract
f,2y
f,1y
m,m
m,1y
f,1y
m,m
m,2y

then we can draw a matrix 
gender      contract 
f,m         m,iy,2y

e fill with 1s and zeros correspnding to the row values above

for numerical attributes dictvectorizer will leave them intact

In [45]:
from sklearn.feature_extraction import DictVectorizer

In [46]:
df_train[['gender','contract']].iloc[:10]

,gender,contract
0,female,two_year
1,male,month-to-month
2,female,month-to-month
3,female,month-to-month
4,female,two_year
5,male,month-to-month
6,male,month-to-month
7,female,month-to-month
8,female,two_year
9,female,month-to-month


In [ ]:
train_dicts=df_train[categorical+numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

DictVectorizer(sparse=False)

In [54]:
val_dicts = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts) #so that we do not fit on val dataset

LOGISTIC REGRESSION

In [55]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy}")